# Category error rate (v2)

Produces the figure that shows the category with a conspicuous error rate.

## Changes from `kusa_category_error_outlier`

| # | Place | Change | Reason |
|---|---|---|---|
| 1 | Setup | Paths from `config.py`, no `files.upload()` fallback | the files live fixed in the project tree |
| 2 | Denominator | Category totals from `dev_pool.csv` instead of `KurdiSent.csv` | 20% are in the test set; otherwise the rates would be about a fifth too low |
| 3 | Numerator | Errors from `cv/<variant>/oof_predictions.csv` instead of the misclassified export | no dependency on an intermediate format |
| 4 | new | Wilson confidence interval per category | makes visible which bars are reliable |
| 5 | new | Categories below the minimum size are hatched in the plot | Technology drops to about 126 rows in the dev pool |
| 6 | Output | to `analysis/` instead of the working directory | |

All changed places are marked with `# [V2]`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ===== [V2-CHANGE 1] Paths from config.py =====
# Original: ORIG_PATH = "KurdiSent.csv" / MIS_PATH = "..._misclassified.csv"
#           plus a files.upload() fallback when the file was missing.
import sys, os, json
# --- locate the project root -------------------------------------------
# No hardcoded Drive path: take KUSA_ROOT if it is set, otherwise the first
# candidate that actually contains config.py. Works in Colab and locally.
import os, sys
_CANDIDATES = [
    os.environ.get("KUSA_ROOT", ""),
    "/content/drive/MyDrive/v2_heldout",
    "/content/drive/MyDrive/google_colab/kusa/v2_heldout",
    os.getcwd(),
    os.path.dirname(os.getcwd()),
]
V2_ROOT = next((p for p in _CANDIDATES
                if p and os.path.isfile(os.path.join(p, "config.py"))), None)
assert V2_ROOT, ("config.py not found - set KUSA_ROOT to the v2_heldout "
                 "directory, e.g. os.environ['KUSA_ROOT'] = '/content/drive/MyDrive/v2_heldout'")
sys.path.insert(0, V2_ROOT)
print("project root:", V2_ROOT)
from config import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt

VARIANT   = "baseline_v2"        # [V2] which variant is evaluated
MIN_N     = 200                  # [V2] minimum size for a reliable rate
print("Variant:", VARIANT)
# ===== End [V2-CHANGE 1] =====

In [ ]:
# ===== [V2-CHANGE 2+3] Denominator from the dev pool, numerator from the OOF predictions =====
dev = pd.read_csv(DEV_POOL, encoding="utf-8")                       # [V2]
oof = pd.read_csv(os.path.join(cv_dir(VARIANT), "oof_predictions.csv"),
                  encoding="utf-8")                                  # [V2]

d = dev[["row_id", "category"]].merge(oof[["row_id", "label", "pred"]],
                                      on="row_id", how="inner")
assert len(d) == len(dev), "OOF predictions do not fully cover the dev pool"
d["correct"] = d["label"] == d["pred"]

print(f"Dev pool: {len(d)} rows | errors: {int((~d['correct']).sum())}")
print("NOTE: the denominator is the dev pool, not the full corpus. The rates are"
      " comparable to the numbers of the old evaluation, but the absolute"
      " error counts are not.")
# ===== End [V2-CHANGE 2+3] =====

In [ ]:
# ===== [V2-CHANGE 4] Wilson CI per category =====
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan, np.nan)
    p = k / n
    den = 1 + z*z/n
    c = (p + z*z/(2*n)) / den
    h = (z * sqrt(p*(1-p)/n + z*z/(4*n*n))) / den
    return p, max(0.0, c - h), min(1.0, c + h)

rows = []
for cat, sub in d.groupby("category"):
    n   = len(sub)
    err = int((~sub["correct"]).sum())
    p, lo, hi = wilson(err, n)
    rows.append({"category": cat, "total": n, "errors": err,
                 "error_rate_%": round(100*p, 2),
                 "ci_lo_%": round(100*lo, 2), "ci_hi_%": round(100*hi, 2),
                 "ci_width_pp": round(100*(hi-lo), 2),
                 "reliable": n >= MIN_N})

tbl = pd.DataFrame(rows).sort_values("error_rate_%", ascending=False).reset_index(drop=True)

n_err_total  = int((~d["correct"]).sum())
overall_rate = 100 * n_err_total / len(d)
reliable = tbl[tbl["reliable"]]
outlier_cat = reliable.loc[reliable["error_rate_%"].idxmax(), "category"]   # [V2] reliable only

print(tbl.to_string(index=False))
print(f"\nOverall error rate: {overall_rate:.1f}%")
print(f"Outlier (reliable categories, n >= {MIN_N}): {outlier_cat}")

small = tbl[~tbl["reliable"]]
if len(small):
    print("\nToo small for a reliable rate:")
    for _, r in small.iterrows():
        print(f"  {r['category']:12s} n={int(r['total']):4d}  "
              f"{r['error_rate_%']:.1f}%  (CI {r['ci_lo_%']:.1f}-{r['ci_hi_%']:.1f}, "
              f"width {r['ci_width_pp']:.1f} pp)")

tbl.to_csv(os.path.join(ANALYSIS, f"category_error_rates_{VARIANT}.csv"),
           index=False, encoding="utf-8")
# ===== End [V2-CHANGE 4] =====

In [ ]:
plot_df = tbl.sort_values("error_rate_%", ascending=True).reset_index(drop=True)
outlier_idx = int(plot_df.index[plot_df["category"] == outlier_cat][0])

HIGHLIGHT, MUTED, INK = "#c62828", "#b9c2cc", "#37474f"

fig, ax = plt.subplots(figsize=(9, 5))
colors = [HIGHLIGHT if i == outlier_idx else MUTED for i in range(len(plot_df))]

bars = ax.barh(plot_df["category"].str.capitalize(), plot_df["error_rate_%"],
               color=colors, edgecolor="white", height=0.68)

# [V2] change 5: hatch unreliable categories
for i, rel in enumerate(plot_df["reliable"]):
    if not rel:
        bars[i].set_hatch("///")
        bars[i].set_edgecolor("#78909c")

# [V2] change 4: error bars from the Wilson CI
ax.errorbar(plot_df["error_rate_%"], range(len(plot_df)),
            xerr=[plot_df["error_rate_%"] - plot_df["ci_lo_%"],
                  plot_df["ci_hi_%"] - plot_df["error_rate_%"]],
            fmt="none", ecolor=INK, elinewidth=1.1, capsize=3, alpha=0.75)

ax.axvline(overall_rate, color=INK, ls="--", lw=1.3, zorder=0)
ax.text(overall_rate + 0.3, -0.62, f"Overall error rate: {overall_rate:.1f}%",
        color=INK, fontsize=9, va="center")

for i, (r, n, hi, rel) in enumerate(zip(plot_df["error_rate_%"], plot_df["total"],
                                        plot_df["ci_hi_%"], plot_df["reliable"])):
    is_out = (i == outlier_idx)
    suffix = "" if rel else "  *"
    ax.text(hi + 0.6, i, f"{r:.1f}%   (n={n:,}){suffix}", va="center", fontsize=9.5,
            color=HIGHLIGHT if is_out else "#546e7a",
            fontweight="bold" if is_out else "normal")

ax.annotate("Outlier", xy=(plot_df["error_rate_%"].iloc[outlier_idx] - 3, outlier_idx),
            fontsize=11, fontweight="bold", color="white", va="center", ha="center")

ax.set_xlabel("Misclassification rate (%)", fontsize=11)
ax.set_title("Misclassification rate by category\n"
             f"{VARIANT} - 5-fold CV, dev-pool out-of-fold (n={len(d):,})",   # [V2] title
             fontsize=13, fontweight="bold", loc="left")
ax.set_xlim(0, plot_df["ci_hi_%"].max() + 8)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", ls=":", alpha=0.4)
if (~plot_df["reliable"]).any():
    ax.text(0.0, -1.25, f"*  n < {MIN_N}: rate unreliable (hatched)",
            fontsize=8.5, color="#78909c", transform=ax.get_yaxis_transform())

plt.tight_layout()
out_png = os.path.join(ANALYSIS, f"category_error_outlier_{VARIANT}.png")   # [V2]
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.show()
print("saved:", out_png)

In [ ]:
assert_test_untouched(globals())
print("\nThe figure is based on the dev pool. If the category claim in the"
      " paper should additionally be confirmed on the test set, that happens"
      " in test_evaluation_v2 - not here.")